In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

class Attacker(nn.Module):
    def __init__(self, input_dim, hidden_dim=300):
        super(Attacker, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, 2)  # Binary Gender (0 or 1)
        )

    def forward(self, x):
        return self.net(x)

In [12]:
def get_dataset_features(loader, num_verbs, device):
    """
    Extracts Ground Truth features (One-Hot Verbs) and Gender labels.
    """
    all_features = []
    all_genders = []
    
    for _, verbs, genders in tqdm(loader, desc="Extracting Dataset Features", leave=False):
        # Convert integer verbs to One-Hot float tensors
        one_hot = F.one_hot(verbs, num_classes=num_verbs).float()
        
        all_features.append(one_hot)
        all_genders.append(genders)
        
    return torch.cat(all_features).to(device), torch.cat(all_genders).to(device)

def get_model_features(baseline_model, loader, device):
    """
    Extracts Model Logits (features) and Gender labels.
    """
    baseline_model.eval()
    all_logits = []
    all_genders = []
    
    with torch.no_grad():
        for images, _, genders in tqdm(loader, desc="Extracting Model Logits", leave=False):
            images = images.to(device)
            
            # Forward pass to get logits (before Softmax)
            logits = baseline_model(images)
            
            all_logits.append(logits)
            all_genders.append(genders)
            
    return torch.cat(all_logits).to(device), torch.cat(all_genders).to(device)

In [13]:
from tqdm import tqdm

def train_attacker(X_train, y_train, X_test, y_test, device, name="Attacker"):
    # Create datasets
    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)
    
    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    # Initialize attacker model
    input_dim = X_train.shape[1]
    model = Attacker(input_dim=input_dim).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Standard LR for MLP
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    epochs = 20  # MLPs converge quickly
    
    # ---- Epoch loop with tqdm ----
    for epoch in tqdm(range(epochs), desc=f"{name} Training"):
        model.train()

        # ---- Batch loop with tqdm ----
        train_iter = tqdm(train_dl, desc=f"Epoch {epoch+1}", leave=False)

        for features, targets in train_iter:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        
        # ---- Evaluation ----
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for features, targets in test_dl:
                outputs = model(features)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
    
    print(f"{name} Results | Best Accuracy: {best_acc:.2f}%")
    return best_acc

In [14]:
from pathlib import Path

IMSITU_PATH = Path("/content/imsitu_dataset/")

import json

with open(IMSITU_PATH / "imsitu_annotated_train.json", "r") as f:
    train_data = json.load(f)

with open(IMSITU_PATH / "imsitu_annotated_dev.json", "r") as f:
    val_data = json.load(f)

train_verbs = set([item['verb'] for item in train_data])
val_verbs = set([item['verb'] for item in val_data])

all_verbs = train_verbs.union(val_verbs)
verb_to_idx = {v: i for i, v in enumerate(sorted(list(all_verbs)))} 

In [15]:
from PIL import Image
import json
from torch.utils.data import Dataset

class ImsituDataset(Dataset):
    def __init__(self, source, transform=None, verb2idx=None):
        self.transform = transform

        with open(source, "r") as f:
            self.data = json.load(f)

        # We keep this for reference, but we don't rely on it for model dimension
        self.verbs = sorted(set(i['verb'] for i in self.data))
        self.verb_to_idx = verb2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load Image
        img_path = IMSITU_PATH / "of500_images_resized" / item['image_path']
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        # FIX 1: Get the verb string first, then map it to the index
        verb_str = item['verb']
        verb = self.verb_to_idx[verb_str] 
        
        # FIX 2: Convert "M"/"F" strings to 0/1 integers for future leakage steps
        # "M" -> 0, "F" -> 1
        gender_str = item['gender']
        gender = 0 if gender_str == "M" else 1

        return img, verb, gender

In [16]:
from torchvision import transforms

BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_train.json", transform=train_transform, verb2idx=verb_to_idx)
val_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_dev.json", transform=val_transform, verb2idx=verb_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training on {len(train_dataset)} images. Validating on {len(val_dataset)} images.")

Training on 33603 images. Validating on 11211 images.


In [17]:
num_verbs = len(verb_to_idx)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. CALCULATE DATASET LEAKAGE (Lambda_D) ---
print("\n--- Calculating Dataset Leakage (Lambda_D) ---")
# Extract One-Hot Vectors
d_X_train, d_y_train = get_dataset_features(train_loader, num_verbs, DEVICE)
d_X_val, d_y_val = get_dataset_features(val_loader, num_verbs, DEVICE)

# Train Attacker on Ground Truth
lambda_d = train_attacker(d_X_train, d_y_train, d_X_val, d_y_val, DEVICE, name="Dataset Leakage")


--- Calculating Dataset Leakage (Lambda_D) ---


Dataset Leakage Training: 100%|██████████| 20/20 [00:42<00:00,  2.14s/it]       

Dataset Leakage Results | Best Accuracy: 68.18%


In [18]:
def get_model_features(baseline_model, loader, device):
    """
    Extracts Model Logits (features) and Gender labels.
    """
    baseline_model.eval()
    all_logits = []
    all_genders = []
    
    with torch.no_grad():
        for images, _, genders in tqdm(loader, desc="Extracting Model Logits", leave=False):
            images = images.to(device)
            
            # Forward pass to get logits (before Softmax)
            logits = baseline_model(images)
            
            all_logits.append(logits)
            all_genders.append(genders)
            
    return torch.cat(all_logits).to(device), torch.cat(all_genders).to(device)

## Model Leakage

In [22]:
import torch.nn as nn
import torchvision.models as models

class BaselineResNet(nn.Module):
    def __init__(self, num_verbs):
        super(BaselineResNet, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        feature_dim = resnet.fc.in_features
        self.classifier = nn.Linear(feature_dim, num_verbs)

    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        logits = self.classifier(f)
        return logits

## Baseline (UnBiased Model)

In [23]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = BaselineResNet(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("baseline_resnet_imsitu.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


BaselineResNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Con

In [24]:
print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:42<00:00,  2.14s/it]     

Model Leakage Results | Best Accuracy: 73.78%


In [25]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.18%
Model Leakage   (Lambda_M): 73.78%
Bias Amplification (Delta): 5.60%


## Debiased Model (`adv@conv4`)

In [ ]:
num_verbs = len(verb_to_idx)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. CALCULATE DATASET LEAKAGE (Lambda_D) ---
print("\n--- Calculating Dataset Leakage (Lambda_D) ---")
# Extract One-Hot Vectors
d_X_train, d_y_train = get_dataset_features(train_loader, num_verbs, DEVICE)
d_X_val, d_y_val = get_dataset_features(val_loader, num_verbs, DEVICE)

# Train Attacker on Ground Truth
lambda_d = train_attacker(d_X_train, d_y_train, d_X_val, d_y_val, DEVICE, name="Dataset Leakage")

In [ ]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = BaselineResNet(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("debaised_resnet_adv@conv4.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

In [ ]:
print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")

In [ ]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)